# $(SASA) Models - Kmeans$

In [1]:
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
import pickle
import numpy as np
import pandas as pd

from pmbrl.model2 import Base_Line_Simple_Model
# from pmbrl.model2 import Model, Regularized_Reference_Loss
from pmbrl.data import Experiment_Data, get_data_expanded

In [2]:
def evaluate(models, df, data):
    def predict(model):
        prediction_dataset = df.copy()
        prediction_dataset[model.grouped_targets_lables] = prediction_dataset.apply(lambda row: data._predict_from_row(row, model), axis=1, result_type='expand')
        return prediction_dataset

    predictions = [predict(m) for m in models]

    prediction_dataset = df.copy()
    for i, pred in enumerate(predictions):
        prediction_dataset[f'estimated_s_model_{i}'] = pred['estimated_s']
        
        results = data.get_evaluation_metrics(pred, p=False)
        prediction_dataset[f'rse_model_{i}'] = results['rse']
        prediction_dataset[f'rse_normalized_model_{i}'] = results['rse_normalized']

        prediction_dataset[f'rse_s0_model_{i}'] = results['rse_s0']
        prediction_dataset[f'rse_s1_model_{i}'] = results['rse_s1']
        prediction_dataset[f'rse_s2_model_{i}'] = results['rse_s2']
        prediction_dataset[f'rse_s3_model_{i}'] = results['rse_s3']

        prediction_dataset[f'rse_s0_normalized_model_{i}'] = results['rse_s0_normalized']
        prediction_dataset[f'rse_s1_normalized_model_{i}'] = results['rse_s1_normalized']
        prediction_dataset[f'rse_s2_normalized_model_{i}'] = results['rse_s2_normalized']
        prediction_dataset[f'rse_s3_normalized_model_{i}'] = results['rse_s3_normalized']


    return prediction_dataset

In [3]:
def reagroup(prediction_dataset, models):
    # agg_results = prediction_dataset[['episode'] + [
    #     f'rse_model_{i}' for i,_ in enumerate(models)
    # ]].groupby('episode').mean().reset_index()
    agg_results = prediction_dataset[['episode', 'step'] + [
        f'rse_model_{i}' for i,_ in enumerate(models)
    ]].copy()
    
    agg_results['best_model'] = agg_results.apply(lambda row: np.argmin(row[2:].values), axis=1)
    print(agg_results['best_model'].value_counts())

    prediction_dataset['best_model'] = prediction_dataset.apply(
        lambda row: agg_results.loc[(agg_results['episode']==row['episode']) & (agg_results['step']==row['step'])].best_model.values[0],
        axis=1
    )
    prediction_dataset['best_rse'] = prediction_dataset.apply(lambda row: row[f'rse_model_{row.best_model}'],axis=1)

    return prediction_dataset

In [4]:
def predicts(models, df, data):
    pre_df = df.copy()
    pre_df[['s_0', 's_1', 's_2', 's_3', 'a_', 's__0', 's__1', 's__2', 's__3']] = pre_df[['s0', 's1', 's2', 's3', 'a', 's_0', 's_1', 's_2', 's_3']]

    prediction_dataset = evaluate(models, pre_df, data)
    prediction_dataset = reagroup(prediction_dataset, models)
    final_predictions = evaluate(models, df, data)
    final_predictions['group'] = prediction_dataset['best_model']

    cols = [
        'estimated_s', 'rse', 'rse_normalized',
        'rse_s0', 'rse_s1', 'rse_s2', 'rse_s3', 
        'rse_s0_normalized', 'rse_s1_normalized',
        'rse_s2_normalized', 'rse_s3_normalized'
    ]

    for c in cols:
        final_predictions[c] = final_predictions.apply(lambda x: x[f'{c}_model_{x.group}'], axis=1)

    return final_predictions[cols]

In [ ]:
rses = []
rses_norm = []

for m in range(10):
    print('model: ', m)
    nome_do_arquivo = f'kmodels_{m+1}.pkl'

    with open(nome_do_arquivo, 'rb') as arquivo:
        exp = pickle.load(arquivo)
        data = exp['data']
        models = exp['model']

    del data
    del exp
    del arquivo

    data = Experiment_Data()
    data.load(path='../testing_data.csv')

    expansions = {
        's': ['s0', 's1', 's2', 's3'],
        's_': ['s_0', 's_1', 's_2', 's_3'],
        's__': ['s__0', 's__1', 's__2', 's__3']
    }
    df = get_data_expanded(data.build_training_dataset(), expansions)
    prediction_dataset = predicts(models, df, data)
    rses.append(prediction_dataset.rse.mean())
    rses_norm.append(prediction_dataset.rse_normalized.mean())
    

best_model
3    792
2    481
4    348
1    305
0    201
Name: count, dtype: int64
best_model
0    928
3    405
2    393
4    292
1    109
Name: count, dtype: int64
best_model
2    916
0    418
1    362
4    221
3    210
Name: count, dtype: int64
best_model
1    989
4    486
0    257
2    252
3    143
Name: count, dtype: int64
best_model
4    991
2    471
0    308
3    231
1    126
Name: count, dtype: int64
best_model
2    590
1    448
0    424
3    358
4    307
Name: count, dtype: int64
best_model
1    960
4    431
0    284
2    262
3    190
Name: count, dtype: int64
best_model
3    1041
4     466
2     273
0     219
1     128
Name: count, dtype: int64
best_model
2    808
3    430
1    370
4    368
0    151
Name: count, dtype: int64
best_model
3    1030
2     450
0     270
4     232
1     145
Name: count, dtype: int64


In [6]:
print('rse:', sum(rses)/len(rses))
print('rses_norm:', sum(rses_norm)/len(rses_norm))

rse: 0.2580945463093559
rses_norm: 0.5139348229432534
